# Week 5, Day 3: Linear systems and inverses: recovering causes from effects

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w005/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gaussian elimination

Complete `eliminate(A, b)`. For each column, swap up the row with the largest entry in that column (partial pivoting), then subtract multiples of the pivot row from the rows below. Finish with back substitution.

In [ ]:
import numpy as np

def eliminate(A, b):
    M = np.column_stack([A.astype(float), b.astype(float)])   # augmented matrix [A | b]
    n = len(b)
    for c in range(n):
        p = c + np.argmax(np.abs(M[c:, c]))                    # pivot row
        M[[c, p]] = M[[p, c]]                                  # swap
        for r in range(c + 1, n):
            ...                                                # subtract a multiple of row c
    x = np.zeros(n)
    for r in range(n - 1, -1, -1):
        x[r] = ...                                             # back substitution
    return x

print(eliminate(np.array([[2.0, 1.0], [1.0, 3.0]]), np.array([4.0, 7.0])))

In [ ]:
import numpy as np
def test_small():
    "solves the worked example"
    assert np.allclose(eliminate(np.array([[2.0, 1.0], [1.0, 3.0]]), np.array([4.0, 7.0])), [1.0, 2.0])

def test_random():
    "matches np.linalg.solve on random systems"
    rng = np.random.default_rng(0)
    for _ in range(5):
        A = rng.normal(size=(4, 4)); b = rng.normal(size=4)
        assert np.allclose(eliminate(A, b), np.linalg.solve(A, b))

def test_swap():
    "handles a zero in the first pivot position by swapping rows"
    A = np.array([[0.0, 1.0], [1.0, 1.0]]); b = np.array([2.0, 3.0])
    assert np.allclose(eliminate(A, b), [1.0, 2.0])

test_small()
test_random()
test_swap()
print("All checks passed")

### Exercise: Unmix three neurons from three electrodes

Three neurons produce signals `S` (3 by 1000 time points). Three electrodes record `Y = M @ S`, where the mixing matrix `M` says how strongly each electrode picks up each neuron. Recover the sources `S_hat` from `Y` using `np.linalg.solve` (it accepts a matrix of right-hand sides).

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
t = np.linspace(0, 1, 1000)
S = np.vstack([np.sin(2 * np.pi * 7 * t), np.sign(np.sin(2 * np.pi * 3 * t)), (t * 5) % 1 - 0.5])
M = np.array([[1.0, 0.6, 0.3], [0.4, 1.0, 0.5], [0.2, 0.5, 1.0]])
Y = M @ S

S_hat = ...
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
axes[0].plot(t, Y.T); axes[0].set_title("what the electrodes record")
axes[1].plot(t, S_hat.T); axes[1].set_title("recovered sources"); axes[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_recovered():
    "S_hat recovers the source signals"
    assert S_hat.shape == S.shape
    assert np.allclose(S_hat, S, atol=1e-8)

test_recovered()
print("All checks passed")

### Exercise: Invisible activity

Two muscles read three motor-cortex neurons through the 2 by 3 matrix `A`. Find a non-zero vector `n` with $A\mathbf{n} = \mathbf{0}$. In three dimensions, the cross product of the two rows (`np.cross`) is orthogonal to both, so each row's dot product with it is zero. Then show that adding `5 * n` to the activity `x` changes the neurons but not the muscles.

In [ ]:
A = np.array([[1.0, 2.0, 0.5], [0.0, 1.0, -1.0]])
x = np.array([3.0, 1.0, 2.0])
n = ...
print("A @ n =", np.round(A @ n, 10))
print("muscles before:", A @ x, " after adding 5n:", A @ (x + 5 * n))
print("neurons before:", x, " after:", x + 5 * n)

In [ ]:
import numpy as np
def test_null():
    "n is a non-zero vector in the null space of A"
    assert np.linalg.norm(n) > 1e-6 and np.allclose(A @ n, 0)

def test_invisible():
    "adding the null vector leaves the readout unchanged"
    assert np.allclose(A @ (x + 5 * n), A @ x)
    assert np.linalg.norm((x + 5 * n) - x) > 1

test_null()
test_invisible()
print("All checks passed")

In [ ]:
for eps in [1.0, 0.1, 0.01]:
    A = np.array([[1.0, 1.0], [1.0, 1.0 + eps]])      # columns become nearly parallel as eps shrinks
    b = np.array([2.0, 2.0 + eps])
    x = np.linalg.solve(A, b)
    x_noisy = np.linalg.solve(A, b + np.array([0.0, 0.001]))   # 0.1 per cent noise in one measurement
    print(f"eps={eps:5}: cond = {np.linalg.cond(A):8.1f}   x = {np.round(x, 3)}   with tiny noise: {np.round(x_noisy, 3)}")